<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise: Integrate Memory into Your Agent System_sol </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

# Exercise: Integrate Memory into Your Agent System

**Purpose:**  
Add long-term memory to your agent so it can remember past conversations and provide better, contextual answers.

**Real-Life Scenario:**  
You are building a **Personal Investment Research Agent** that helps retail investors research tech and AI stocks, track their portfolio, and get personalized advice over multiple sessions.

## 1. Setup

In [1]:
# Run this first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*" faiss-cpu beautifulsoup4 requests --force-reinstall
from datetime import datetime
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print(" Setup complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.

In [2]:
from langchain_core.tools import tool
import requests
from bs4 import BeautifulSoup
@tool
def get_stock_info(company: str) -> str:
    """Fetch real company overview, AI projects, profit drivers, and SDG alignment."""
    company = company.lower()
    data = {
        "microsoft": "Microsoft leads in cloud AI (Azure AI, Copilot). Main profit driver: Cloud + Enterprise software. SDG alignment: SDG 9, SDG 13.",
        "nvidia": "NVIDIA dominates AI hardware with GPUs. Main projects: CUDA, DGX. Profit driver: Data center GPUs. SDG alignment: SDG 9, SDG 13.",
        "google": "Google (Alphabet) excels in search AI and Gemini. Profit driver: Advertising + Cloud. SDG alignment: SDG 9, SDG 13.",
        "amazon": "Amazon leads in e-commerce AI and AWS. Main projects: Bedrock, SageMaker. SDG alignment: SDG 9, SDG 12."
    }
    return data.get(company, f"Information for {company} not available in demo.")

In [3]:
#Long-Term Memory with Vector Store
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(["Initial empty memory"], embeddings)

def store_interaction(query: str, response: str):
    doc = Document(
        page_content=f"User: {query}\nAssistant: {response}",
        metadata={"timestamp": str(datetime.now())}
    )
    vectorstore.add_documents([doc])
    print(f"Stored in long-term memory: {query[:60]}...")

def retrieve_relevant_memory(query: str, k: int = 3):
    docs = vectorstore.similarity_search(query, k=k)
    return "\n\n".join([doc.page_content for doc in docs])

In [4]:
tools = [get_stock_info]

In [5]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langsmith import Client

client = Client()


llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Use the standard ReAct prompt (no custom {history} in template)
prompt = client.pull_prompt("hwchase17/react",
    dangerously_pull_public_prompt=True)

# Create the agent
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=12,
    handle_parsing_errors=True
)

print("Agent created with standard ReAct prompt")

Agent created with standard ReAct prompt


In [6]:
queries = [
    "Tell me about Microsoft's main AI projects and which SDG goals they follow.",
    "What are NVIDIA's key strengths and sustainability efforts?",
    "Compare the values and long-term vision of Google and Amazon in AI."
]

for q in queries:
    print(f"\n{'='*80}")
    print(f"User: {q}")

    # Retrieve relevant past memory from vector store
    past_memory = retrieve_relevant_memory(q)
    if past_memory:
        print(f"(Retrieved from long-term memory)\n{past_memory}\n")

    # Pass memory into the agent via the {history} slot
    response = executor.invoke({
        "input": q,
        "history": past_memory or "No relevant past interactions found."
    })
    # # Run the agent with the current query
    # response = executor.invoke({"input": q})

    print("Agent:", response["output"])

    # Store this interaction for future long-term retrieval
    store_interaction(q, response["output"])


User: Tell me about Microsoft's main AI projects and which SDG goals they follow.
(Retrieved from long-term memory)
Initial empty memory



> Entering new AgentExecutor chain...
I should use the get_stock_info tool to gather information on Microsoft's AI projects and SDG alignment.
Action: get_stock_info
Action Input: "Microsoft"Microsoft leads in cloud AI (Azure AI, Copilot). Main profit driver: Cloud + Enterprise software. SDG alignment: SDG 9, SDG 13.I have the information needed to answer the question.
Final Answer: Microsoft's main AI projects include Azure AI and Copilot. They align with SDG 9 (Industry, Innovation, and Infrastructure) and SDG 13 (Climate Action).

> Finished chain.
Agent: Microsoft's main AI projects include Azure AI and Copilot. They align with SDG 9 (Industry, Innovation, and Infrastructure) and SDG 13 (Climate Action).
Stored in long-term memory: Tell me about Microsoft's main AI projects and which SDG goa...

User: What are NVIDIA's key strengths and sustai